# Módulo 2, Parte 1. pandas para investigación social: lectura, auditoría y limpieza inicial de bases

**Curso:** Fundamentos de Programación para IA Generativa Aplicada  
**Orientación:** Ciencias sociales aplicadas  
**Sesión:** 2, Parte 1

---

## Propósito de esta parte

En esta sesión introducimos `pandas` como la librería central para trabajar con datos tabulares en investigación social aplicada. El objetivo no es solo “abrir archivos”, sino aprender a: leer bases de datos de forma reproducible; identificar la unidad de observación y las llaves empíricas; auditar tipos, missings y consistencia básica; limpiar variables numéricas y códigos especiales; y construir un subconjunto limpio, listo para análisis posterior.

---

## Idea central

En ciencias sociales, limpiar datos no es una tarea secundaria.  
Es el momento en que definimos qué observaciones son válidas, qué columnas representan realmente una variable y qué transformaciones conservan la estructura del problema. En otras palabras, un `DataFrame` es un objeto tabular; una limpieza correcta es una transformación controlada; una llave bien tratada preserva identidad observacional; y una base mal tipada destruye interpretabilidad antes de cualquier análisis.

## 1. Marco conceptual mínimo

Una base de datos social no es un espejo transparente de la realidad.  
Es una construcción institucional. Por ello, antes de cualquier análisis, debemos responder cuatro preguntas:

1. **¿Cuál es la unidad de observación?**  
   ¿Persona, hogar, distrito, departamento?

2. **¿Qué identificadores preservan esa unidad?**  
   En ENAHO, por ejemplo, una persona suele identificarse con una llave compuesta.

3. **¿Qué variables observadas son analíticamente relevantes?**  
   No toda columna es necesaria para la pregunta de investigación.

4. **¿Qué problemas de estructura tiene la base?**  
   Tipos erróneos, códigos especiales, ceros perdidos, duplicados, missings.

Este módulo parte de una idea metodológica simple:  
**sin estructura empírica clara, no hay análisis sustantivo confiable.**

In [1]:
# Montar Google Drive

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Librerías principales

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 2.2.2
numpy : 2.0.2


In [4]:
# Ruta principal de trabajo
# Ajusta solo si tu carpeta tiene una variación mínima en nombre o acentos.

RUTA_SESION_2 = Path("/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Insumos/Sesión 2")

if not RUTA_SESION_2.exists():
    raise FileNotFoundError(
        f"No se encontró la ruta: {RUTA_SESION_2}\n"
        "Verifica el nombre exacto de las carpetas en Google Drive."
    )

print("Ruta encontrada correctamente:")
print(RUTA_SESION_2)

Ruta encontrada correctamente:
/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Insumos/Sesión 2


In [5]:
# Inspección inicial de archivos disponibles

archivos = sorted([p.name for p in RUTA_SESION_2.iterdir()])
print(f"Total de archivos/carpetas detectados: {len(archivos)}\n")

for nombre in archivos[:30]:
    print("-", nombre)

Total de archivos/carpetas detectados: 21

- CSE_2021_Enaho01-2021-200.csv
- CSE_2022_Enaho01-2022-200.csv
- CSE_2023_Enaho01-2023-200.csv
- CSE_2024_Enaho01-2024-200.csv
- ECO_2021_Enaho01a-2021-500.csv
- ECO_2022_Enaho01a-2022-500.csv
- ECO_2023_Enaho01a-2023-500.csv
- ECO_2024_Enaho01a-2024-500.csv
- EDU_2021_Enaho01a-2021-300.csv
- EDU_2022_Enaho01a-2022-300.csv
- EDU_2023_Enaho01A-2023-300.csv
- EDU_2024_Enaho01A-2024-300.csv
- PIB_2020_2024_pbi_peru_16.xlsx
- POL_2021_Enaho01-2021-800A.csv
- POL_2022_Enaho01-2022-800A.csv
- POL_2023_Enaho01-2023-800A.csv
- POL_2024_Enaho01-2024-800A.csv
- VID_2021_Enaho01-2021-100.csv
- VID_2022_Enaho01-2022-100.csv
- VID_2023_01-2023-100.csv
- VID_2024_Enaho01-2024-100.csv


## 2. Recordatorio breve: Series y DataFrames

Antes de trabajar con ENAHO, conviene fijar la diferencia estructural: una **Serie** es una columna con índice; un **DataFrame** es una tabla compuesta por múltiples Series alineadas por índice. La Serie es una variable observable organizada; el DataFrame es una estructura de observaciones y atributos. Lo importante no es memorizar la definición, sino entender que casi toda la investigación computacional con encuestas trabaja sobre DataFrames.

In [6]:
# Microejemplo breve y alineado con ciencias sociales

mini = pd.DataFrame({
    "departamento": ["Cusco", "Puno", "Lima"],
    "ingreso_promedio": [1450.0, 1320.0, 2850.0],
    "anios_educacion": [9.4, 8.8, 11.8]
})

print("DataFrame completo:\n")
display(mini)

print("Tipo de mini['ingreso_promedio']:", type(mini["ingreso_promedio"]))
print("Tipo de mini[['ingreso_promedio', 'anios_educacion']]:", type(mini[["ingreso_promedio", "anios_educacion"]]))

DataFrame completo:



,departamento,ingreso_promedio,anios_educacion
0,Cusco,"1,450.00",9.40
1,Puno,"1,320.00",8.80
2,Lima,"2,850.00",11.80


Tipo de mini['ingreso_promedio']: <class 'pandas.core.series.Series'>
Tipo de mini[['ingreso_promedio', 'anios_educacion']]: <class 'pandas.core.frame.DataFrame'>


## 3. Definición de llaves y variables de interés

Para esta primera parte trabajaremos con un subconjunto del módulo de ingresos de ENAHO 2024.

### Llave de identificación de persona

Usaremos las siguientes columnas como identificadores:

- `UBIGEO`
- `CONGLOME`
- `VIVIENDA`
- `HOGAR`
- `CODPERSO`

### Variables sustantivas iniciales

- `D524A1`: ingreso principal
- `D538A1`: ingreso secundario

No cargaremos todo el archivo si no es necesario.  
En investigación reproducible, seleccionar columnas relevantes desde el inicio mejora claridad, velocidad y control analítico.

In [7]:
# Parámetros de trabajo

ARCHIVO_ECO = RUTA_SESION_2 / "ECO_2024_Enaho01a-2024-500.csv"

ID_PERSONA = ["UBIGEO", "CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"]
VARS_INGRESO = ["D524A1", "D538A1"]
COLS_ECO = ID_PERSONA + VARS_INGRESO

if not ARCHIVO_ECO.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo esperado: {ARCHIVO_ECO.name}\n"
        "Verifica el nombre exacto del CSV dentro de la carpeta."
    )

print("Archivo identificado correctamente:")
print(ARCHIVO_ECO.name)

Archivo identificado correctamente:
ECO_2024_Enaho01a-2024-500.csv


In [9]:
# Funciones auxiliares para una práctica más rigurosa

def auditar_dataframe(df: pd.DataFrame, nombre: str = "df") -> pd.DataFrame:
    """
    Devuelve una tabla-resumen con tipo, missings y cardinalidad por variable.
    """
    resumen = pd.DataFrame({
        "variable": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "n_missing": df.isna().sum().values,
        "pct_missing": (df.isna().mean() * 100).round(2).values,
        "n_unicos": df.nunique(dropna=True).values
    })
    print(f"{nombre}: {df.shape[0]:,} filas × {df.shape[1]:,} columnas")
    return resumen.sort_values(["pct_missing", "variable"], ascending=[False, True]).reset_index(drop=True)


def convertir_a_numerico(df: pd.DataFrame, columnas: Iterable[str]) -> pd.DataFrame:
    """
    Convierte columnas a numérico; valores no convertibles pasan a NaN.
    """
    for col in columnas:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def reemplazar_codigos_especiales(df: pd.DataFrame, columnas: Iterable[str], codigo=999999) -> pd.DataFrame:
    """
    Reemplaza un código especial por NaN.
    """
    for col in columnas:
        df[col] = df[col].replace(codigo, np.nan)
    return df


def normalizar_llaves(df: pd.DataFrame, columnas: Iterable[str]) -> pd.DataFrame:
    """
    Homogeneiza identificadores como texto y corrige UBIGEO a 6 dígitos.
    """
    for col in columnas:
        df[col] = df[col].astype("string").str.strip()
    if "UBIGEO" in columnas:
        df["UBIGEO"] = df["UBIGEO"].str.zfill(6)
    return df


def verificar_duplicados(df: pd.DataFrame, llave: list[str]) -> pd.DataFrame:
    """
    Retorna observaciones duplicadas según la llave indicada.
    """
    duplicados = df[df.duplicated(subset=llave, keep=False)].sort_values(llave)
    print(f"Duplicados encontrados en la llave {llave}: {len(duplicados):,}")
    return duplicados

In [10]:
# Carga controlada del archivo
# Solo columnas necesarias, para preservar claridad y eficiencia.

eco = pd.read_csv(
    ARCHIVO_ECO,
    usecols=COLS_ECO,
    sep=",",
    encoding="latin-1",
    low_memory=False
)

print("Base cargada correctamente.")
print(f"Filas    : {eco.shape[0]:,}")
print(f"Columnas : {eco.shape[1]:,}")

display(eco.head())

Base cargada correctamente.
Filas    : 85,992
Columnas : 7


,CONGLOME,VIVIENDA,HOGAR,CODPERSO,UBIGEO,D524A1,D538A1
0,15009,13,11,1,10101,,
1,15009,13,11,2,10101,,
2,15009,47,11,1,10101,11267,
3,15009,47,11,2,10101,6447,
4,15009,59,11,1,10101,,


In [11]:
# Auditoría inicial: estructura antes de limpiar

auditoria_inicial = auditar_dataframe(eco, nombre="eco (raw)")
display(auditoria_inicial)

eco (raw): 85,992 filas × 7 columnas


,variable,dtype,n_missing,pct_missing,n_unicos
0,CODPERSO,int64,0,0.00,16
1,CONGLOME,int64,0,0.00,5359
2,D524A1,object,0,0.00,11842
3,D538A1,object,0,0.00,2019
4,HOGAR,int64,0,0.00,15
5,UBIGEO,int64,0,0.00,1271
6,VIVIENDA,int64,0,0.00,541


## 4. Primera lectura metodológica de la auditoría

Esta auditoría cumple una función analítica, no solo técnica. Nos permite responder: qué columnas son numéricas y cuáles no; qué variables presentan alta proporción de faltantes; si las llaves parecen tener cardinalidad razonable; si debemos intervenir tipos y códigos antes de cualquier descripción. En una encuesta, los faltantes y códigos especiales no son “ruido”: forman parte de la estructura empírica que luego debemos interpretar con cuidado.

In [12]:
# Limpieza 1: convertir ingresos a numérico

eco = convertir_a_numerico(eco, VARS_INGRESO)

print("Tipos luego de convertir columnas de ingreso:")
print(eco[VARS_INGRESO].dtypes)

Tipos luego de convertir columnas de ingreso:
D524A1    float64
D538A1    float64
dtype: object


In [13]:
# Limpieza 2: reemplazar códigos especiales del INEI por NaN
# En ingreso, 999999 suele representar un valor especial no analíticamente interpretable como ingreso real.

eco = reemplazar_codigos_especiales(eco, VARS_INGRESO, codigo=999999)

print("Missings después de reemplazar códigos especiales:")
print(eco[VARS_INGRESO].isna().sum())

Missings después de reemplazar códigos especiales:
D524A1    61023
D538A1    82697
dtype: int64


In [14]:
# Limpieza 3: normalizar llaves e impedir pérdida de estructura en UBIGEO

eco = normalizar_llaves(eco, ID_PERSONA)

# Extraer código de departamento
eco["cod_dpto"] = eco["UBIGEO"].str[:2]

# Indicador simple para Cusco
eco["es_cusco"] = (eco["cod_dpto"] == "08").astype("int8")

display(eco.head())

,CONGLOME,VIVIENDA,HOGAR,CODPERSO,UBIGEO,D524A1,D538A1,cod_dpto,es_cusco
0,15009,13,11,1,010101,NaN,NaN,01,0
1,15009,13,11,2,010101,NaN,NaN,01,0
2,15009,47,11,1,010101,"11,267.00",NaN,01,0
3,15009,47,11,2,010101,"6,447.00",NaN,01,0
4,15009,59,11,1,010101,NaN,NaN,01,0


In [15]:
# Verificación de consistencia de llaves

print("Tipos de las columnas llave:\n")
print(eco[ID_PERSONA].dtypes)

duplicados_eco = verificar_duplicados(eco, ID_PERSONA)

# Mostrar solo si existieran duplicados
if len(duplicados_eco) > 0:
    display(duplicados_eco.head(10))

Tipos de las columnas llave:

UBIGEO      string[python]
CONGLOME    string[python]
VIVIENDA    string[python]
HOGAR       string[python]
CODPERSO    string[python]
dtype: object
Duplicados encontrados en la llave ['UBIGEO', 'CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO']: 0


In [16]:
# Auditoría posterior a la limpieza

auditoria_limpia = auditar_dataframe(eco, nombre="eco (limpio)")
display(auditoria_limpia)

eco (limpio): 85,992 filas × 9 columnas


,variable,dtype,n_missing,pct_missing,n_unicos
0,D538A1,float64,82697,96.17,2017
1,D524A1,float64,61023,70.96,11840
2,CODPERSO,string,0,0.00,16
3,CONGLOME,string,0,0.00,5359
4,HOGAR,string,0,0.00,15
5,UBIGEO,string,0,0.00,1271
6,VIVIENDA,string,0,0.00,541
7,cod_dpto,string,0,0.00,25
8,es_cusco,int8,0,0.00,2


## 5. Construcción de un subconjunto analítico inicial

En lugar de saltar de inmediato a operaciones complejas, conviene definir primero un subconjunto limpio y legible. Aquí haremos dos cosas: conservar una base limpia a nivel persona; y construir un subconjunto de Cusco para la continuidad del módulo. Este paso ya comienza a preparar la trayectoria de la Tarea 1, donde el recorte territorial será clave.

In [17]:
# Subconjunto Cusco

eco_cusco = eco.loc[eco["cod_dpto"] == "08"].copy()

print(f"Observaciones Perú  : {len(eco):,}")
print(f"Observaciones Cusco : {len(eco_cusco):,}")

display(eco_cusco.head())

Observaciones Perú  : 85,992
Observaciones Cusco : 3,140


,CONGLOME,VIVIENDA,HOGAR,CODPERSO,UBIGEO,D524A1,D538A1,cod_dpto,es_cusco
1722,16345,48,11,1,080106,"51,936.00","10,885.00",08,1
1723,16345,48,11,2,080106,"16,933.00",NaN,08,1
1724,16345,48,11,3,080106,NaN,NaN,08,1
1725,16345,66,11,1,080106,"14,514.00",NaN,08,1
1726,16345,66,11,2,080106,NaN,NaN,08,1


In [18]:
# Primer resumen descriptivo comparado: Perú vs Cusco

def resumen_ingresos(df: pd.DataFrame, etiqueta: str) -> pd.Series:
    return pd.Series({
        "n_obs": len(df),
        "n_ingreso_principal_valido": df["D524A1"].notna().sum(),
        "media_ingreso_principal": df["D524A1"].mean(),
        "mediana_ingreso_principal": df["D524A1"].median(),
        "n_ingreso_secundario_valido": df["D538A1"].notna().sum(),
        "media_ingreso_secundario": df["D538A1"].mean(),
        "mediana_ingreso_secundario": df["D538A1"].median()
    }, name=etiqueta)

resumen_comp = pd.concat(
    [
        resumen_ingresos(eco, "Perú"),
        resumen_ingresos(eco_cusco, "Cusco")
    ],
    axis=1
).T.round(2)

display(resumen_comp)

,n_obs,n_ingreso_principal_valido,media_ingreso_principal,mediana_ingreso_principal,n_ingreso_secundario_valido,media_ingreso_secundario,mediana_ingreso_secundario
Perú,"85,992.00","24,969.00","20,462.73","15,636.00","3,295.00","6,725.66","4,187.00"
Cusco,"3,140.00",682.00,"20,858.22","16,681.50",197.00,"10,311.89","7,620.00"


In [19]:
# Tabla corta de cobertura de datos para las variables de ingreso

cobertura = pd.DataFrame({
    "variable": VARS_INGRESO,
    "n_validos_peru": [eco[col].notna().sum() for col in VARS_INGRESO],
    "pct_validos_peru": [(eco[col].notna().mean() * 100) for col in VARS_INGRESO],
    "n_validos_cusco": [eco_cusco[col].notna().sum() for col in VARS_INGRESO],
    "pct_validos_cusco": [(eco_cusco[col].notna().mean() * 100) for col in VARS_INGRESO]
}).round(2)

display(cobertura)

,variable,n_validos_peru,pct_validos_peru,n_validos_cusco,pct_validos_cusco
0,D524A1,24969,29.04,682,21.72
1,D538A1,3295,3.83,197,6.27


## 6. Interpretación inicial

Hasta aquí todavía no estamos explicando causalmente nada.  
Lo que sí hemos logrado es mucho más básico y más importante:

- definir unidad de observación;
- preservar la identidad de cada registro;
- convertir variables a tipos analíticamente útiles;
- tratar códigos especiales de manera explícita;
- construir un subconjunto territorial consistente.

En otras palabras, ya tenemos una base mínima para pasar del archivo bruto al objeto analítico.

In [20]:
# Opcional: crear una variable simple que servirá luego en la Parte 2

eco["ingreso_total_simple"] = eco[["D524A1", "D538A1"]].sum(axis=1, min_count=1)
eco_cusco["ingreso_total_simple"] = eco_cusco[["D524A1", "D538A1"]].sum(axis=1, min_count=1)

display(
    eco_cusco[ID_PERSONA + ["D524A1", "D538A1", "ingreso_total_simple"]].head()
)

,UBIGEO,CONGLOME,VIVIENDA,HOGAR,CODPERSO,D524A1,D538A1,ingreso_total_simple
1722,080106,16345,48,11,1,"51,936.00","10,885.00","62,821.00"
1723,080106,16345,48,11,2,"16,933.00",NaN,"16,933.00"
1724,080106,16345,48,11,3,NaN,NaN,NaN
1725,080106,16345,66,11,1,"14,514.00",NaN,"14,514.00"
1726,080106,16345,66,11,2,NaN,NaN,NaN


In [21]:
# Opcional: guardar subconjuntos limpios para continuidad del trabajo

SALIDA_PERU = RUTA_SESION_2 / "eco_2024_subset_limpio.csv"
SALIDA_CUSCO = RUTA_SESION_2 / "eco_2024_cusco_limpio.csv"

eco.to_csv(SALIDA_PERU, index=False)
eco_cusco.to_csv(SALIDA_CUSCO, index=False)

print("Archivos guardados:")
print("-", SALIDA_PERU.name)
print("-", SALIDA_CUSCO.name)

Archivos guardados:
- eco_2024_subset_limpio.csv
- eco_2024_cusco_limpio.csv


## 7. Ejercicio aplicado

Con la base ya limpia, realiza las siguientes tareas: (i) calcula el porcentaje de observaciones de Cusco con ingreso principal válido; (ii) compara la media y mediana del ingreso principal entre Perú y Cusco; (iii) revisa si existen duplicados en la llave de persona; y (iv) explica por qué `UBIGEO` debe tratarse como texto y no como entero.

### Pregunta metodológica

¿Por qué una limpieza incorrecta no solo produce errores técnicos, sino también errores sustantivos de interpretación?

## 8. Cierre de la Parte 1

En esta primera parte hemos transformado un archivo bruto en una base empíricamente legible.

### Lo ya logrado

- lectura reproducible de archivos desde Drive;
- selección de columnas relevantes;
- auditoría estructural de la base;
- normalización de llaves;
- tratamiento explícito de missings y códigos especiales;
- construcción de un subconjunto territorial para Cusco.

### Lo que sigue en la Parte 2

Sobre este mismo Colab continuaremos con:

- creación de variables más sustantivas;
- `groupby` y agregaciones;
- `concat` para varios años;
- `merge` con otras bases;
- primeros ensamblajes analíticos para la Tarea 1.

La regla metodológica es simple: **primero estructura, luego transformación, después comparación, y solo entonces interpretación.**

# Módulo 2, Parte 2. Transformación, ensamblaje multianual y articulación macroeconómica con ENAHO

**Curso:** Fundamentos de Programación para IA Generativa Aplicada  
**Orientación:** Ciencias sociales aplicadas  
**Sesión:** 2, Parte 2

---

## Propósito de esta parte

En esta segunda parte pasamos de la limpieza inicial a la construcción de una base analítica multianual. El objetivo ya no es solo leer y depurar archivos, sino: construir variables de análisis con criterios explícitos; respetar rigurosamente la diferencia entre nivel persona y nivel hogar; generar una base homogénea para 2021–2024; colapsar esa base a nivel departamento × año; integrar una base externa de PIB departamental; y exportar resultados listos para análisis posterior, con énfasis en Cusco.

---

## Idea central

Un pipeline empírico riguroso no consiste en “pegar tablas”. Consiste en preservar estructura: preservar la identidad de las unidades de observación, distinguir correctamente persona, hogar, departamento y año; transformar variables sin destruir su sentido sustantivo; y documentar el paso de archivo bruto a objeto analítico. Así, `pandas` deja de ser solo una herramienta de manipulación y pasa a ser una forma de formalización empírica.

In [22]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 2.2.2
numpy : 2.0.2


In [24]:
# Rutas de trabajo

RUTA_INSUMOS = Path("/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Insumos/Sesión 2")
RUTA_RESULTADOS = Path("/content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesión 2")

RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

if not RUTA_INSUMOS.exists():
    raise FileNotFoundError(
        f"No se encontró la ruta de insumos:\n{RUTA_INSUMOS}\n"
        "Verifica el nombre exacto de carpetas en tu Google Drive."
    )

print("Ruta de insumos :", RUTA_INSUMOS)
print("Ruta de resultados:", RUTA_RESULTADOS)

Ruta de insumos : /content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Insumos/Sesión 2
Ruta de resultados: /content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesión 2


In [25]:
# Inspección rápida del contenido disponible

archivos = sorted([p.name for p in RUTA_INSUMOS.iterdir()])
print(f"Total de archivos/carpetas detectados: {len(archivos)}\n")

for nombre in archivos[:50]:
    print("-", nombre)

Total de archivos/carpetas detectados: 23

- CSE_2021_Enaho01-2021-200.csv
- CSE_2022_Enaho01-2022-200.csv
- CSE_2023_Enaho01-2023-200.csv
- CSE_2024_Enaho01-2024-200.csv
- ECO_2021_Enaho01a-2021-500.csv
- ECO_2022_Enaho01a-2022-500.csv
- ECO_2023_Enaho01a-2023-500.csv
- ECO_2024_Enaho01a-2024-500.csv
- EDU_2021_Enaho01a-2021-300.csv
- EDU_2022_Enaho01a-2022-300.csv
- EDU_2023_Enaho01A-2023-300.csv
- EDU_2024_Enaho01A-2024-300.csv
- PIB_2020_2024_pbi_peru_16.xlsx
- POL_2021_Enaho01-2021-800A.csv
- POL_2022_Enaho01-2022-800A.csv
- POL_2023_Enaho01-2023-800A.csv
- POL_2024_Enaho01-2024-800A.csv
- VID_2021_Enaho01-2021-100.csv
- VID_2022_Enaho01-2022-100.csv
- VID_2023_01-2023-100.csv
- VID_2024_Enaho01-2024-100.csv
- eco_2024_cusco_limpio.csv
- eco_2024_subset_limpio.csv


## 1. Regla metodológica central: no confundir niveles de observación

En esta parte vamos a trabajar con cuatro niveles distintos: **persona**: ingreso, educación, participación; **hogar**: NBI y agregación de atributos de personas; **departamento**: colapso territorial; **año**: comparación 2021–2024. Si confundimos estos niveles, el código puede correr, pero el análisis queda mal especificado.

### Regla práctica

- los módulos de ingreso, educación y participación se trabajan primero a nivel persona;
- luego se agregan explícitamente a nivel hogar;
- el módulo de NBI opera como ancla del nivel hogar;
- recién después se colapsa a departamento × año.

In [26]:
# Diccionario de archivos por año

ARCHIVOS = {
    2021: {
        "eco": "ECO_2021_Enaho01a-2021-500.csv",
        "edu": "EDU_2021_Enaho01a-2021-300.csv",
        "pol": "POL_2021_Enaho01-2021-800A.csv",
        "vid": "VID_2021_Enaho01-2021-100.csv",
    },
    2022: {
        "eco": "ECO_2022_Enaho01a-2022-500.csv",
        "edu": "EDU_2022_Enaho01a-2022-300.csv",
        "pol": "POL_2022_Enaho01-2022-800A.csv",
        "vid": "VID_2022_Enaho01-2022-100.csv",
    },
    2023: {
        "eco": "ECO_2023_Enaho01a-2023-500.csv",
        "edu": "EDU_2023_Enaho01A-2023-300.csv",
        "pol": "POL_2023_Enaho01-2023-800A.csv",
        "vid": "VID_2023_01-2023-100.csv",
    },
    2024: {
        "eco": "ECO_2024_Enaho01a-2024-500.csv",
        "edu": "EDU_2024_Enaho01A-2024-300.csv",
        "pol": "POL_2024_Enaho01-2024-800A.csv",
        "vid": "VID_2024_Enaho01-2024-100.csv",
    }
}

for anio, d in ARCHIVOS.items():
    print(f"\n{anio}")
    for k, v in d.items():
        print(f"  {k}: {v}")


2021
  eco: ECO_2021_Enaho01a-2021-500.csv
  edu: EDU_2021_Enaho01a-2021-300.csv
  pol: POL_2021_Enaho01-2021-800A.csv
  vid: VID_2021_Enaho01-2021-100.csv

2022
  eco: ECO_2022_Enaho01a-2022-500.csv
  edu: EDU_2022_Enaho01a-2022-300.csv
  pol: POL_2022_Enaho01-2022-800A.csv
  vid: VID_2022_Enaho01-2022-100.csv

2023
  eco: ECO_2023_Enaho01a-2023-500.csv
  edu: EDU_2023_Enaho01A-2023-300.csv
  pol: POL_2023_Enaho01-2023-800A.csv
  vid: VID_2023_01-2023-100.csv

2024
  eco: ECO_2024_Enaho01a-2024-500.csv
  edu: EDU_2024_Enaho01A-2024-300.csv
  pol: POL_2024_Enaho01-2024-800A.csv
  vid: VID_2024_Enaho01-2024-100.csv


In [28]:
# Llaves y variables relevantes

LLAVE_PERSONA = ["UBIGEO", "CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"]
LLAVE_HOGAR = ["UBIGEO", "CONGLOME", "VIVIENDA", "HOGAR"]

VARS_ECO = ["D524A1", "D538A1"]
VARS_EDU = ["P301A", "P301B"]
VARS_POL = [f"P801_{i}" for i in range(1, 21)]
VARS_NBI = ["NBI1", "NBI2", "NBI3", "NBI4", "NBI5"]

COLS_ECO = LLAVE_PERSONA + VARS_ECO
COLS_EDU = LLAVE_PERSONA + VARS_EDU
COLS_POL = LLAVE_PERSONA + VARS_POL
COLS_VID = LLAVE_HOGAR + VARS_NBI

In [29]:
# Llaves y variables relevantes

LLAVE_PERSONA = ["UBIGEO", "CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO"]
LLAVE_HOGAR = ["UBIGEO", "CONGLOME", "VIVIENDA", "HOGAR"]

VARS_ECO = ["D524A1", "D538A1"]
VARS_EDU = ["P301A", "P301B"]
VARS_POL = [f"P801_{i}" for i in range(1, 21)]
VARS_NBI = ["NBI1", "NBI2", "NBI3", "NBI4", "NBI5"]

COLS_ECO = LLAVE_PERSONA + VARS_ECO
COLS_EDU = LLAVE_PERSONA + VARS_EDU
COLS_POL = LLAVE_PERSONA + VARS_POL
COLS_VID = LLAVE_HOGAR + VARS_NBI

In [30]:
# Funciones auxiliares reutilizables

def normalizar_texto(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    x = "".join(
        ch for ch in unicodedata.normalize("NFD", x)
        if unicodedata.category(ch) != "Mn"
    )
    x = re.sub(r"[^a-z0-9 ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def estandarizar_llaves(df, incluir_codperso=False):
    df = df.copy()

    cols = ["UBIGEO", "CONGLOME", "VIVIENDA", "HOGAR"]
    if incluir_codperso and "CODPERSO" in df.columns:
        cols.append("CODPERSO")

    for col in [c for c in cols if c in df.columns]:
        df[col] = df[col].astype("string").str.strip()

    if "UBIGEO" in df.columns:
        df["UBIGEO"] = (
            df["UBIGEO"]
            .str.replace(".0", "", regex=False)
            .str.zfill(6)
        )
        df["cod_dpto"] = df["UBIGEO"].str[:2]

    return df


def convertir_numerico(df, columnas, codigos_missing=None):
    df = df.copy()
    for col in columnas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            if codigos_missing is not None:
                df[col] = df[col].replace(codigos_missing, np.nan)
    return df


def sum_min1(s):
    return s.sum(min_count=1)


def contar_duplicados(df, llave):
    return int(df.duplicated(subset=llave).sum())


def auditar_basico(df, nombre="df"):
    print(f"{nombre}: {df.shape[0]:,} filas × {df.shape[1]:,} columnas")
    print("Duplicados por llave hogar :", contar_duplicados(df, LLAVE_HOGAR if set(LLAVE_HOGAR).issubset(df.columns) else df.columns[:1].tolist()))
    display(df.head())

In [31]:
# Funciones de construcción de variables

def nivel_educ_3cat(cod):
    if pd.isna(cod):
        return np.nan
    cod = int(cod)
    if cod <= 4:
        return "basica_o_menos"
    elif cod <= 6:
        return "secundaria"
    else:
        return "superior"


def procesar_educacion(df_edu):
    df_edu = df_edu.copy()
    df_edu = convertir_numerico(df_edu, VARS_EDU, codigos_missing=[99])

    df_edu["nivel_educ_3cat"] = df_edu["P301A"].apply(nivel_educ_3cat)

    # Indicadores ordinales mínimos y prudentes
    df_edu["tiene_superior"] = np.where(df_edu["P301A"] >= 7, 1.0, np.where(df_edu["P301A"].isna(), np.nan, 0.0))
    df_edu["tiene_universitaria"] = np.where(df_edu["P301A"] >= 9, 1.0, np.where(df_edu["P301A"].isna(), np.nan, 0.0))

    return df_edu


def procesar_ingresos(df_eco):
    df_eco = df_eco.copy()
    df_eco = convertir_numerico(df_eco, VARS_ECO, codigos_missing=[999999])

    df_eco = df_eco.rename(columns={
        "D524A1": "ingreso_principal",
        "D538A1": "ingreso_secundario"
    })

    df_eco["ingreso_persona"] = df_eco[["ingreso_principal", "ingreso_secundario"]].sum(axis=1, min_count=1)
    df_eco["persona_con_ingreso_obs"] = df_eco["ingreso_persona"].notna().astype(float)

    return df_eco


def procesar_participacion(df_pol):
    df_pol = df_pol.copy()
    df_pol = convertir_numerico(df_pol, VARS_POL, codigos_missing=[99])

    # P801_19 suele ser "no participa"; no debe entrar como organización activa
    vars_activas = [c for c in VARS_POL if c != "P801_19" and c in df_pol.columns]

    for col in vars_activas:
        df_pol[col] = np.where(df_pol[col] > 0, 1.0, np.where(df_pol[col].isna(), np.nan, 0.0))

    if "P801_19" in df_pol.columns:
        df_pol["P801_19"] = np.where(df_pol["P801_19"] > 0, 1.0, np.where(df_pol["P801_19"].isna(), np.nan, 0.0))

    df_pol["n_org_persona"] = df_pol[vars_activas].sum(axis=1, min_count=1)
    df_pol["participa_persona"] = np.where(df_pol[vars_activas].sum(axis=1, min_count=1) > 0, 1.0, 0.0)

    return df_pol


def procesar_nbi(df_vid):
    df_vid = df_vid.copy()
    df_vid = convertir_numerico(df_vid, VARS_NBI)

    df_vid["n_nbi"] = df_vid[VARS_NBI].sum(axis=1, min_count=1)
    df_vid["pobre_nbi"] = np.where(df_vid["n_nbi"] >= 1, 1.0, np.where(df_vid["n_nbi"].isna(), np.nan, 0.0))

    return df_vid

## 2. Nota crítica sobre educación

En esta reconstrucción **no** usaré `P301B` como si fuera una medida homogénea de años de escolaridad acumulada. ¿Por qué? Porque `P301B` expresa el año o grado dentro del nivel, y mezclarlo directamente entre primaria, secundaria y superior puede inducir errores sustantivos. Por ahora, lo metodológicamente más sólido es construir: una variable ordinal de nivel; dummies prudentes de educación superior; y agregaciones de hogar basadas en máximos o presencia. Luego podemos construir una variable de años equivalentes con apoyo explícito del diccionario del módulo.

In [36]:
# Función principal de procesamiento por año

def procesar_anio_enaho(anio):
    print("\n" + "=" * 70)
    print(f"PROCESANDO ENAHO {anio}")
    print("=" * 70)

    archivos = ARCHIVOS[anio]

    eco = pd.read_csv(RUTA_INSUMOS / archivos["eco"], sep=",", encoding="latin-1", low_memory=False, usecols=lambda c: c in COLS_ECO)
    edu = pd.read_csv(RUTA_INSUMOS / archivos["edu"], sep=",", encoding="latin-1", low_memory=False, usecols=lambda c: c in COLS_EDU)
    pol = pd.read_csv(RUTA_INSUMOS / archivos["pol"], sep=",", encoding="latin-1", low_memory=False, usecols=lambda c: c in COLS_POL)
    vid = pd.read_csv(RUTA_INSUMOS / archivos["vid"], sep=",", encoding="latin-1", low_memory=False, usecols=lambda c: c in COLS_VID)

    print("Dimensiones raw:")
    print(" eco:", eco.shape, "| edu:", edu.shape, "| pol:", pol.shape, "| vid:", vid.shape)

    eco = estandarizar_llaves(eco, incluir_codperso=True)
    edu = estandarizar_llaves(edu, incluir_codperso=True)
    pol = estandarizar_llaves(pol, incluir_codperso=True)
    vid = estandarizar_llaves(vid, incluir_codperso=False)

    eco = procesar_ingresos(eco)
    edu = procesar_educacion(edu)
    pol = procesar_participacion(pol)
    vid = procesar_nbi(vid)

    print("\nDuplicados por llave:")
    print(" eco persona:", contar_duplicados(eco, LLAVE_PERSONA))
    print(" edu persona:", contar_duplicados(edu, LLAVE_PERSONA))
    pol_llave_persona = [col for col in LLAVE_PERSONA if col in pol.columns]
    print(" pol persona:", contar_duplicados(pol, pol_llave_persona))
    print(" vid hogar  :", contar_duplicados(vid, LLAVE_HOGAR))

    # Base persona: ingresos + educación
    base_persona = pd.merge(
        eco,
        edu[LLAVE_PERSONA + ["P301A", "P301B", "nivel_educ_3cat", "tiene_superior", "tiene_universitaria"]],
        on=LLAVE_PERSONA,
        how="left"
    )

    # Agregación persona -> hogar (ingresos y educación observados en módulos de persona)
    hogar_ing_edu = (
        base_persona
        .groupby(LLAVE_HOGAR, as_index=False)
        .agg(
            cod_dpto=("cod_dpto", "first"),
            n_registros_ingreso=("CODPERSO", "nunique"),
            ingreso_principal_total=("ingreso_principal", sum_min1),
            ingreso_secundario_total=("ingreso_secundario", sum_min1),
            ingreso_hogar_total=("ingreso_persona", sum_min1),
            hogar_con_superior=("tiene_superior", "max"),
            hogar_con_universitaria=("tiene_universitaria", "max"),
            max_nivel_educ=("P301A", "max")
        )
    )

    hogar_ing_edu["ln_ingreso_hogar"] = np.where(
        hogar_ing_edu["ingreso_hogar_total"].notna() & (hogar_ing_edu["ingreso_hogar_total"] >= 0),
        np.log1p(hogar_ing_edu["ingreso_hogar_total"]),
        np.nan
    )

    # Agregación persona -> hogar (participación)
    pol_agg_dict = {
        "cod_dpto": ("cod_dpto", "first"),
        "n_org_hogar": ("n_org_persona", sum_min1),
        "hogar_participa": ("participa_persona", "max"),
        "tasa_participacion_obs": ("participa_persona", "mean")
    }
    if "CODPERSO" in pol.columns:
        pol_agg_dict["n_registros_participacion"] = ("CODPERSO", "nunique")

    pol_hogar = (
        pol
        .groupby(LLAVE_HOGAR, as_index=False)
        .agg(**pol_agg_dict)
    )

    # Ancla del hogar: módulo de condiciones de vida
    vid_hogar = (
        vid
        .groupby(LLAVE_HOGAR, as_index=False)
        .agg(
            cod_dpto=("cod_dpto", "first"),
            NBI1=("NBI1", "first"),
            NBI2=("NBI2", "first"),
            NBI3=("NBI3", "first"),
            NBI4=("NBI4", "first"),
            NBI5=("NBI5", "first"),
            n_nbi=("n_nbi", "first"),
            pobre_nbi=("pobre_nbi", "first")
        )
    )

    # Merge final a nivel hogar
    base_hogar = (
        vid_hogar
        .merge(
            hogar_ing_edu.drop(columns=["cod_dpto"]),
            on=LLAVE_HOGAR,
            how="left",
            validate="1:1"
        )
        .merge(
            pol_hogar.drop(columns=["cod_dpto"]),
            on=LLAVE_HOGAR,
            how="left",
            validate="1:1"
        )
    )

    base_hogar["anio"] = anio

    print("\nBase hogar resultante:")
    print(f" filas   : {len(base_hogar):,}")
    print(f" columnas: {base_hogar.shape[1]}")
    print(" hogares con ingreso observado :", int(base_hogar["ingreso_hogar_total"].notna().sum()))
    print(" hogares con participación obs :", int(base_hogar["hogar_participa"].notna().sum()))
    print(" hogares pobres por NBI        :", int(base_hogar["pobre_nbi"].sum(skipna=True)))

    return base_hogar

In [37]:
# Ejecutar para 2021–2024

bases_hogar = {}

for anio in [2021, 2022, 2023, 2024]:
    bases_hogar[anio] = procesar_anio_enaho(anio)



PROCESANDO ENAHO 2021
Dimensiones raw:
 eco: (86806, 7) | edu: (109867, 7) | pol: (32731, 24) | vid: (43524, 9)

Duplicados por llave:
 eco persona: 0
 edu persona: 0
 pol persona: 0
 vid hogar  : 0

Base hogar resultante:
 filas   : 43,524
 columnas: 24
 hogares con ingreso observado : 18468
 hogares con participación obs : 32731
 hogares pobres por NBI        : 5846

PROCESANDO ENAHO 2022
Dimensiones raw:
 eco: (87661, 7) | edu: (110257, 7) | pol: (34213, 24) | vid: (44122, 9)

Duplicados por llave:
 eco persona: 0
 edu persona: 0
 pol persona: 0
 vid hogar  : 0

Base hogar resultante:
 filas   : 44,122
 columnas: 24
 hogares con ingreso observado : 19441
 hogares con participación obs : 34213
 hogares pobres por NBI        : 5846

PROCESANDO ENAHO 2023
Dimensiones raw:
 eco: (86654, 7) | edu: (108354, 7) | pol: (33886, 24) | vid: (44378, 9)

Duplicados por llave:
 eco persona: 0
 edu persona: 0
 pol persona: 0
 vid hogar  : 0

Base hogar resultante:
 filas   : 44,378
 columnas: 24


In [38]:
# Apilar años

enaho_hogar_panel = pd.concat(
    [bases_hogar[anio] for anio in [2021, 2022, 2023, 2024]],
    ignore_index=True
)

print("Panel hogar ENAHO 2021–2024")
print(f"Filas totales : {len(enaho_hogar_panel):,}")
print(f"Columnas      : {enaho_hogar_panel.shape[1]}")
print("\nHogares por año:")
print(enaho_hogar_panel["anio"].value_counts().sort_index())

display(enaho_hogar_panel.head())

Panel hogar ENAHO 2021–2024
Filas totales : 176,755
Columnas      : 24

Hogares por año:
anio
2021    43524
2022    44122
2023    44378
2024    44731
Name: count, dtype: int64


,UBIGEO,CONGLOME,VIVIENDA,HOGAR,cod_dpto,NBI1,NBI2,NBI3,NBI4,NBI5,n_nbi,pobre_nbi,n_registros_ingreso,ingreso_principal_total,ingreso_secundario_total,ingreso_hogar_total,hogar_con_superior,hogar_con_universitaria,max_nivel_educ,ln_ingreso_hogar,n_org_hogar,hogar_participa,tasa_participacion_obs,anio
0,010101,10400,15,11,01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.000,NaN,"3,166.000","3,166.000",0.000,0.000,6.000,8.061,3.000,1.000,1.000,2021
1,010101,10400,22,11,01,0.000,1.000,0.000,0.000,0.000,1.000,1.000,3.000,"10,924.277",NaN,"10,924.277",0.000,0.000,6.000,9.299,0.000,0.000,0.000,2021
2,010101,10400,34,11,01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.000,"7,003.312",NaN,"7,003.312",1.000,1.000,10.000,8.854,2.000,1.000,1.000,2021
3,010101,10400,39,11,01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.000,NaN,NaN,NaN,0.000,0.000,5.000,NaN,0.000,0.000,0.000,2021
4,010101,10400,51,11,01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,4.000,"33,347.421",NaN,"33,347.421",1.000,0.000,8.000,10.415,0.000,0.000,0.000,2021


## 3. Qué representa exactamente esta base hogar × año

Esta base no representa “todos los atributos posibles del hogar”. Representa un hogar observado en el módulo de condiciones de vida, al cual se le han anexado: agregados de ingreso provenientes del módulo 500; agregados educativos prudentes provenientes del módulo 300; agregados de participación provenientes del módulo 800A.

Por eso algunas variables pueden tener faltantes: no todo hogar tiene necesariamente observación completa en todos los módulos individuales. Esa no es una “falla” del dataset; es parte de su estructura empírica.

In [39]:
# Colapso a departamento × año

panel_dpto = (
    enaho_hogar_panel
    .groupby(["cod_dpto", "anio"], as_index=False)
    .agg(
        n_hogares=("UBIGEO", "count"),
        pct_hogares_ingreso_obs=("ingreso_hogar_total", lambda s: s.notna().mean()),
        ingreso_prom_hogar=("ingreso_hogar_total", "mean"),
        ingreso_mediano_hogar=("ingreso_hogar_total", "median"),
        ln_ingreso_prom_hogar=("ln_ingreso_hogar", "mean"),
        pct_hogares_con_superior=("hogar_con_superior", "mean"),
        pct_hogares_con_universitaria=("hogar_con_universitaria", "mean"),
        pct_hogares_participa=("hogar_participa", "mean"),
        n_org_prom_hogar=("n_org_hogar", "mean"),
        pct_pobre_nbi=("pobre_nbi", "mean"),
        nbi_prom=("n_nbi", "mean")
    )
    .round(4)
)

print("Base departamento × año:")
print(f"Filas: {len(panel_dpto)}")
display(panel_dpto.head(12))

Base departamento × año:
Filas: 100


,cod_dpto,anio,n_hogares,pct_hogares_ingreso_obs,ingreso_prom_hogar,ingreso_mediano_hogar,ln_ingreso_prom_hogar,pct_hogares_con_superior,pct_hogares_con_universitaria,pct_hogares_participa,n_org_prom_hogar,pct_pobre_nbi,nbi_prom
0,01,2021,1516,0.413,"16,707.210","9,677.985",9.162,0.290,0.151,0.664,0.864,0.272,0.360
1,01,2022,1553,0.394,"17,362.056","11,487.500",9.273,0.274,0.146,0.688,0.922,0.282,0.356
2,01,2023,1548,0.399,"16,481.181","11,139.000",9.251,0.291,0.138,0.736,1.119,0.294,0.387
3,01,2024,1583,0.389,"18,200.894","10,849.000",9.264,0.286,0.142,0.690,1.052,0.298,0.382
4,02,2021,1748,0.432,"19,872.371","13,840.217",9.333,0.378,0.255,0.470,0.791,0.157,0.172
5,02,2022,1788,0.429,"22,298.978","15,237.000",9.493,0.379,0.265,0.486,0.857,0.168,0.184
6,02,2023,1839,0.414,"22,695.050","15,913.500",9.574,0.397,0.254,0.525,0.954,0.179,0.195
7,02,2024,1845,0.413,"26,868.073","19,162.500",9.743,0.409,0.265,0.538,1.005,0.151,0.161
8,03,2021,1083,0.416,"17,916.139","12,883.748",9.265,0.300,0.192,0.740,0.961,0.120,0.131
9,03,2022,1098,0.447,"21,584.177","16,236.000",9.481,0.305,0.206,0.759,0.978,0.107,0.108


In [40]:
# Mapa de códigos departamentales para lectura posterior

MAPA_COD_DPTO = {
    "01": "Amazonas",
    "02": "Áncash",
    "03": "Apurímac",
    "04": "Arequipa",
    "05": "Ayacucho",
    "06": "Cajamarca",
    "07": "Callao",
    "08": "Cusco",
    "09": "Huancavelica",
    "10": "Huánuco",
    "11": "Ica",
    "12": "Junín",
    "13": "La Libertad",
    "14": "Lambayeque",
    "15": "Lima",
    "16": "Loreto",
    "17": "Madre de Dios",
    "18": "Moquegua",
    "19": "Pasco",
    "20": "Piura",
    "21": "Puno",
    "22": "San Martín",
    "23": "Tacna",
    "24": "Tumbes",
    "25": "Ucayali"
}

panel_dpto["departamento"] = panel_dpto["cod_dpto"].map(MAPA_COD_DPTO)
display(panel_dpto.head())

,cod_dpto,anio,n_hogares,pct_hogares_ingreso_obs,ingreso_prom_hogar,ingreso_mediano_hogar,ln_ingreso_prom_hogar,pct_hogares_con_superior,pct_hogares_con_universitaria,pct_hogares_participa,n_org_prom_hogar,pct_pobre_nbi,nbi_prom,departamento
0,01,2021,1516,0.413,"16,707.210","9,677.985",9.162,0.290,0.151,0.664,0.864,0.272,0.360,Amazonas
1,01,2022,1553,0.394,"17,362.056","11,487.500",9.273,0.274,0.146,0.688,0.922,0.282,0.356,Amazonas
2,01,2023,1548,0.399,"16,481.181","11,139.000",9.251,0.291,0.138,0.736,1.119,0.294,0.387,Amazonas
3,01,2024,1583,0.389,"18,200.894","10,849.000",9.264,0.286,0.142,0.690,1.052,0.298,0.382,Amazonas
4,02,2021,1748,0.432,"19,872.371","13,840.217",9.333,0.378,0.255,0.470,0.791,0.157,0.172,Áncash


## 4. Integración del PIB departamental

Ahora agregaremos una base externa de PIB departamental. Aquí el punto metodológico importante es que el PIB no reemplaza a la microdata. Su función es distinta: la ENAHO describe hogares y personas; el PIB describe el contexto agregado territorial. Por tanto, el merge correcto es a nivel **departamento × año**, no a nivel persona ni hogar.

In [41]:
# Cargar base de PIB

ARCHIVO_PIB = RUTA_INSUMOS / "PIB_2020_2024_pbi_peru_16.xlsx"

if not ARCHIVO_PIB.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo de PIB:\n{ARCHIVO_PIB}"
    )

pib_raw = pd.read_excel(ARCHIVO_PIB, skiprows=6)
pib_raw = pib_raw.rename(columns={pib_raw.columns[0]: "departamento_raw"})
pib_raw = pib_raw[pib_raw["departamento_raw"].notna()].reset_index(drop=True)

print("Columnas detectadas:")
print(list(pib_raw.columns))
display(pib_raw.head(15))

Columnas detectadas:
['departamento_raw', 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, '2022P/', '2023P/', '2024E/']


,departamento_raw,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022P/,2023P/,2024E/
0,Amazonas,"1,778,775.000","1,930,947.000","2,058,318.000","2,210,682.000","2,287,107.000","2,551,601.000","2,682,266.000","2,824,603.000","2,782,128.000","2,784,366.000","2,940,822.000","3,118,373.000","3,168,990.000","3,030,350.000","3,122,772.000","3,064,074.000","3,124,439.000","3,103,841.000"
1,Áncash,"15,672,771.000","16,854,588.000","16,400,826.000","16,013,215.000","16,155,687.000","17,666,947.000","18,478,843.000","16,028,265.000","17,584,621.000","18,365,696.000","19,317,454.000","20,712,339.000","20,059,093.000","18,758,235.000","21,466,272.000","21,721,407.000","20,931,135.000","21,386,810.000"
2,Apurímac,"1,824,181.000","1,688,564.000","1,623,801.000","1,765,744.000","1,869,417.000","2,110,908.000","2,342,674.000","2,437,434.000","2,630,345.000","6,343,065.000","7,718,535.000","7,131,314.000","7,170,478.000","6,448,415.000","6,586,239.000","6,290,806.000","6,689,337.000","7,219,959.000"
3,Arequipa,"16,991,831.000","18,885,807.000","19,032,479.000","20,158,733.000","21,038,813.000","22,033,542.000","22,629,103.000","22,773,308.000","23,524,592.000","29,623,112.000","30,724,797.000","31,506,818.000","31,404,343.000","26,489,345.000","29,970,944.000","31,539,363.000","31,374,274.000","31,636,176.000"
4,Ayacucho,"2,975,676.000","3,401,175.000","3,750,401.000","3,922,514.000","4,111,349.000","4,482,971.000","4,906,299.000","4,879,476.000","5,162,331.000","5,177,917.000","5,451,854.000","5,760,202.000","5,931,518.000","5,193,538.000","5,796,822.000","5,996,577.000","5,913,691.000","6,212,319.000"
5,Cajamarca,"8,159,499.000","9,319,769.000","10,050,467.000","10,140,905.000","10,595,497.000","11,270,583.000","11,086,928.000","10,855,588.000","10,798,612.000","10,581,305.000","10,901,682.000","11,209,419.000","11,479,756.000","10,310,549.000","11,488,958.000","11,904,977.000","11,885,309.000","12,001,887.000"
6,Cusco,"10,913,725.000","11,663,686.000","13,631,820.000","15,405,459.000","17,384,466.000","17,711,332.000","20,708,699.000","20,723,581.000","21,071,852.000","21,898,270.000","21,576,717.000","21,700,735.000","22,006,880.000","19,284,962.000","20,592,202.000","21,489,757.000","22,507,388.000","22,767,865.000"
7,Huancavelica,"2,475,279.000","2,613,850.000","2,696,095.000","2,817,536.000","2,909,215.000","3,143,661.000","3,174,927.000","3,281,748.000","3,265,820.000","3,212,948.000","3,354,985.000","3,525,421.000","3,527,812.000","3,279,521.000","3,493,040.000","3,433,141.000","3,513,289.000","3,912,627.000"
8,Huánuco,"3,200,861.000","3,464,132.000","3,499,798.000","3,739,082.000","3,955,589.000","4,380,310.000","4,642,728.000","4,799,787.000","5,114,983.000","5,345,445.000","5,832,171.000","6,010,056.000","6,081,484.000","5,437,276.000","5,972,022.000","6,225,057.000","6,600,610.000","6,924,290.000"
9,Ica,"8,793,956.000","10,415,637.000","10,841,974.000","11,607,992.000","12,883,432.000","13,067,505.000","14,394,675.000","14,809,397.000","15,295,581.000","15,325,191.000","16,206,741.000","16,994,391.000","17,656,354.000","15,612,472.000","19,614,370.000","20,877,156.000","20,766,164.000","21,439,833.000"


In [42]:
# Selección robusta de columnas 2021–2024

def seleccionar_columnas_pib(df):
    cols = list(df.columns)
    year_map = {}

    for target in [2021, 2022, 2023, 2024]:
        candidatos = [c for c in cols if str(target) in str(c)]
        if len(candidatos) == 0:
            raise KeyError(f"No se encontró columna para el año {target} en la base de PIB.")
        year_map[candidatos[0]] = target

    out = df[["departamento_raw"] + list(year_map.keys())].copy()
    out = out.rename(columns=year_map)
    return out

pib_sel = seleccionar_columnas_pib(pib_raw)
display(pib_sel.head(10))

,departamento_raw,2021,2022,2023,2024
0,Amazonas,"3,122,772.000","3,064,074.000","3,124,439.000","3,103,841.000"
1,Áncash,"21,466,272.000","21,721,407.000","20,931,135.000","21,386,810.000"
2,Apurímac,"6,586,239.000","6,290,806.000","6,689,337.000","7,219,959.000"
3,Arequipa,"29,970,944.000","31,539,363.000","31,374,274.000","31,636,176.000"
4,Ayacucho,"5,796,822.000","5,996,577.000","5,913,691.000","6,212,319.000"
5,Cajamarca,"11,488,958.000","11,904,977.000","11,885,309.000","12,001,887.000"
6,Cusco,"20,592,202.000","21,489,757.000","22,507,388.000","22,767,865.000"
7,Huancavelica,"3,493,040.000","3,433,141.000","3,513,289.000","3,912,627.000"
8,Huánuco,"5,972,022.000","6,225,057.000","6,600,610.000","6,924,290.000"
9,Ica,"19,614,370.000","20,877,156.000","20,766,164.000","21,439,833.000"


In [43]:
# Normalizar nombres departamentales y mapear a cod_dpto

pib_sel["departamento_norm"] = pib_sel["departamento_raw"].apply(normalizar_texto)

MAPA_NOMBRE_A_COD = {
    "amazonas": "01",
    "ancash": "02",
    "apurimac": "03",
    "arequipa": "04",
    "ayacucho": "05",
    "cajamarca": "06",
    "prov const del callao": "07",
    "callao": "07",
    "cusco": "08",
    "huancavelica": "09",
    "huanuco": "10",
    "ica": "11",
    "junin": "12",
    "la libertad": "13",
    "lambayeque": "14",
    "lima": "15",
    "loreto": "16",
    "madre de dios": "17",
    "moquegua": "18",
    "pasco": "19",
    "piura": "20",
    "puno": "21",
    "san martin": "22",
    "tacna": "23",
    "tumbes": "24",
    "ucayali": "25"
}

# Excluir subdivisiones que no corresponden al mismo nivel territorial del panel departamental
EXCLUIR_NOMBRES = {
    "region lima",
    "provincia de lima",
    "lima metropolitana"
}

pib_sel = pib_sel[~pib_sel["departamento_norm"].isin(EXCLUIR_NOMBRES)].copy()
pib_sel["cod_dpto"] = pib_sel["departamento_norm"].map(MAPA_NOMBRE_A_COD)

sin_codigo = pib_sel[pib_sel["cod_dpto"].isna()]["departamento_raw"].unique()
print("Departamentos sin código asignado:")
print(sin_codigo if len(sin_codigo) > 0 else "Ninguno")

Departamentos sin código asignado:
['Valor Agregado Bruto' 'Impuestos a los Productos'
 'Derechos de Importación' 'Producto Bruto Interno'
 'Fuente: Instituto Nacional de Estadística e Informática'
 'Con información disponible al 15 de Noviembre del 2025']


In [47]:
# Transformar a formato largo y calcular crecimiento

for col in [2021, 2022, 2023, 2024]:
    pib_sel[col] = pd.to_numeric(pib_sel[col], errors="coerce")

pib_long = pib_sel.melt(
    id_vars=["departamento_raw", "departamento_norm", "cod_dpto"],
    value_vars=[2021, 2022, 2023, 2024],
    var_name="anio",
    value_name="pib"
)

pib_long["anio"] = pib_long["anio"].astype(int)
# Aggregate PIB for each department-year combination to resolve duplicates
pib_long = pib_long.groupby(["cod_dpto", "anio"])["pib"].sum().reset_index()

pib_long = pib_long.sort_values(["cod_dpto", "anio"]).reset_index(drop=True)

pib_long["pib_crecimiento"] = pib_long.groupby("cod_dpto")["pib"].pct_change() * 100

# Duplicados por departamento × año
dup_pib = pib_long.duplicated(subset=["cod_dpto", "anio"]).sum()
print("Duplicados en PIB por cod_dpto × año:", dup_pib)

display(pib_long.head(12))

Duplicados en PIB por cod_dpto × año: 0


,cod_dpto,anio,pib,pib_crecimiento
0,01,2021,"3,122,772.000",NaN
1,01,2022,"3,064,074.000",-1.880
2,01,2023,"3,124,439.000",1.970
3,01,2024,"3,103,841.000",-0.659
4,02,2021,"21,466,272.000",NaN
5,02,2022,"21,721,407.000",1.189
6,02,2023,"20,931,135.000",-3.638
7,02,2024,"21,386,810.000",2.177
8,03,2021,"6,586,239.000",NaN
9,03,2022,"6,290,806.000",-4.486


In [48]:
# Merge final: departamento × año + PIB

base_final = panel_dpto.merge(
    pib_long[["cod_dpto", "anio", "pib", "pib_crecimiento"]],
    on=["cod_dpto", "anio"],
    how="left",
    validate="1:1"
)

print("Resumen merge final:")
print(" panel_dpto :", len(panel_dpto))
print(" base_final :", len(base_final))
print(" missings PIB:", int(base_final["pib"].isna().sum()))

display(base_final.head(12))

Resumen merge final:
 panel_dpto : 100
 base_final : 100
 missings PIB: 0


,cod_dpto,anio,n_hogares,pct_hogares_ingreso_obs,ingreso_prom_hogar,ingreso_mediano_hogar,ln_ingreso_prom_hogar,pct_hogares_con_superior,pct_hogares_con_universitaria,pct_hogares_participa,n_org_prom_hogar,pct_pobre_nbi,nbi_prom,departamento,pib,pib_crecimiento
0,01,2021,1516,0.413,"16,707.210","9,677.985",9.162,0.290,0.151,0.664,0.864,0.272,0.360,Amazonas,"3,122,772.000",NaN
1,01,2022,1553,0.394,"17,362.056","11,487.500",9.273,0.274,0.146,0.688,0.922,0.282,0.356,Amazonas,"3,064,074.000",-1.880
2,01,2023,1548,0.399,"16,481.181","11,139.000",9.251,0.291,0.138,0.736,1.119,0.294,0.387,Amazonas,"3,124,439.000",1.970
3,01,2024,1583,0.389,"18,200.894","10,849.000",9.264,0.286,0.142,0.690,1.052,0.298,0.382,Amazonas,"3,103,841.000",-0.659
4,02,2021,1748,0.432,"19,872.371","13,840.217",9.333,0.378,0.255,0.470,0.791,0.157,0.172,Áncash,"21,466,272.000",NaN
5,02,2022,1788,0.429,"22,298.978","15,237.000",9.493,0.379,0.265,0.486,0.857,0.168,0.184,Áncash,"21,721,407.000",1.189
6,02,2023,1839,0.414,"22,695.050","15,913.500",9.574,0.397,0.254,0.525,0.954,0.179,0.195,Áncash,"20,931,135.000",-3.638
7,02,2024,1845,0.413,"26,868.073","19,162.500",9.743,0.409,0.265,0.538,1.005,0.151,0.161,Áncash,"21,386,810.000",2.177
8,03,2021,1083,0.416,"17,916.139","12,883.748",9.265,0.300,0.192,0.740,0.961,0.120,0.131,Apurímac,"6,586,239.000",NaN
9,03,2022,1098,0.447,"21,584.177","16,236.000",9.481,0.305,0.206,0.759,0.978,0.107,0.108,Apurímac,"6,290,806.000",-4.486


In [49]:
# Subconjunto específico para Cusco

base_cusco = base_final[base_final["cod_dpto"] == "08"].sort_values("anio").copy()

print("Serie 2021–2024 para Cusco:")
display(base_cusco)

Serie 2021–2024 para Cusco:


,cod_dpto,anio,n_hogares,pct_hogares_ingreso_obs,ingreso_prom_hogar,ingreso_mediano_hogar,ln_ingreso_prom_hogar,pct_hogares_con_superior,pct_hogares_con_universitaria,pct_hogares_participa,n_org_prom_hogar,pct_pobre_nbi,nbi_prom,departamento,pib,pib_crecimiento
28,08,2021,1643,0.344,"21,037.003","14,841.702",9.478,0.367,0.228,0.642,0.716,0.110,0.129,Cusco,"20,592,202.000",NaN
29,08,2022,1655,0.367,"21,960.449","15,342.000",9.535,0.368,0.231,0.659,0.736,0.109,0.117,Cusco,"21,489,757.000",4.359
30,08,2023,1658,0.365,"22,846.755","18,057.000",9.637,0.381,0.231,0.689,0.820,0.109,0.121,Cusco,"22,507,388.000",4.735
31,08,2024,1765,0.350,"26,348.057","19,272.000",9.756,0.411,0.246,0.709,0.912,0.104,0.112,Cusco,"22,767,865.000",1.157


## 5. Lectura sustantiva mínima

Con esta base ya podemos empezar a formular preguntas analíticas iniciales como: ¿cómo evoluciona el ingreso promedio del hogar en Cusco entre 2021 y 2024? ¿cómo se mueve la pobreza por NBI en ese mismo período? ¿qué pasa con la participación organizativa de los hogares? ¿cómo dialogan esos movimientos con la trayectoria del PIB departamental? Todavía no estamos estimando causalidad. Pero ya construimos un objeto empírico suficientemente sólido para empezar una comparación rigurosa.

In [50]:
# Exportar resultados

ARCHIVOS_SALIDA = {
    "enaho_hogar_panel": RUTA_RESULTADOS / "enaho_hogar_panel_2021_2024.csv",
    "panel_dpto": RUTA_RESULTADOS / "panel_dpto_2021_2024.csv",
    "base_final": RUTA_RESULTADOS / "base_final_2021_2024_pib.csv",
    "base_cusco": RUTA_RESULTADOS / "base_cusco_2021_2024.csv"
}

enaho_hogar_panel.to_csv(ARCHIVOS_SALIDA["enaho_hogar_panel"], index=False, encoding="utf-8")
panel_dpto.to_csv(ARCHIVOS_SALIDA["panel_dpto"], index=False, encoding="utf-8")
base_final.to_csv(ARCHIVOS_SALIDA["base_final"], index=False, encoding="utf-8")
base_cusco.to_csv(ARCHIVOS_SALIDA["base_cusco"], index=False, encoding="utf-8")

print("Archivos exportados correctamente:\n")
for nombre, ruta in ARCHIVOS_SALIDA.items():
    print(f"- {nombre}: {ruta.name}")

Archivos exportados correctamente:

- enaho_hogar_panel: enaho_hogar_panel_2021_2024.csv
- panel_dpto: panel_dpto_2021_2024.csv
- base_final: base_final_2021_2024_pib.csv
- base_cusco: base_cusco_2021_2024.csv


In [51]:
# Resumen final

print("=" * 70)
print("RESUMEN FINAL")
print("=" * 70)

print("\n1) Base hogar × año")
print(f"Filas    : {len(enaho_hogar_panel):,}")
print(f"Columnas : {enaho_hogar_panel.shape[1]}")

print("\n2) Base departamento × año")
print(f"Filas    : {len(panel_dpto):,}")
print(f"Años     : {sorted(panel_dpto['anio'].unique())}")
print(f"Dptos    : {panel_dpto['cod_dpto'].nunique()}")

print("\n3) Base final con PIB")
print(f"Filas    : {len(base_final):,}")
print(f"Missings PIB : {int(base_final['pib'].isna().sum())}")

print("\n4) Subconjunto Cusco")
print(f"Filas    : {len(base_cusco):,}")
display(base_cusco)

RESUMEN FINAL

1) Base hogar × año
Filas    : 176,755
Columnas : 24

2) Base departamento × año
Filas    : 100
Años     : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Dptos    : 25

3) Base final con PIB
Filas    : 100
Missings PIB : 0

4) Subconjunto Cusco
Filas    : 4


,cod_dpto,anio,n_hogares,pct_hogares_ingreso_obs,ingreso_prom_hogar,ingreso_mediano_hogar,ln_ingreso_prom_hogar,pct_hogares_con_superior,pct_hogares_con_universitaria,pct_hogares_participa,n_org_prom_hogar,pct_pobre_nbi,nbi_prom,departamento,pib,pib_crecimiento
28,08,2021,1643,0.344,"21,037.003","14,841.702",9.478,0.367,0.228,0.642,0.716,0.110,0.129,Cusco,"20,592,202.000",NaN
29,08,2022,1655,0.367,"21,960.449","15,342.000",9.535,0.368,0.231,0.659,0.736,0.109,0.117,Cusco,"21,489,757.000",4.359
30,08,2023,1658,0.365,"22,846.755","18,057.000",9.637,0.381,0.231,0.689,0.820,0.109,0.121,Cusco,"22,507,388.000",4.735
31,08,2024,1765,0.350,"26,348.057","19,272.000",9.756,0.411,0.246,0.709,0.912,0.104,0.112,Cusco,"22,767,865.000",1.157


## Cierre del módulo 2

En esta sesión completamos la transición desde archivos brutos hacia una base analítica multianual estructurada.

### Logros del módulo 2 completo

- lectura reproducible desde Google Drive;
- selección de variables relevantes;
- limpieza inicial rigurosa;
- construcción prudente de variables de ingreso, educación, participación y NBI;
- agregación correcta de persona a hogar;
- ensamblaje 2021–2024;
- colapso a departamento × año;
- integración con PIB departamental;
- exportación final para análisis posterior.

### Qué queda preparado para la Tarea 1

Ya tenemos, especialmente para Cusco, una base suficiente para comenzar:

- análisis descriptivo comparado;
- construcción de gráficos;
- formulación de preguntas sustantivas;
- discusión entre condiciones de vida, participación, educación, ingresos y contexto macroeconómico.

La secuencia metodológica queda así:

**archivo bruto → limpieza → variable → hogar → territorio → comparación temporal → interpretación**